# MLP in PyTorch: Non-Linear Binary Classification

This notebook reimplements the one-hidden-layer MLP from my *from-scratch*
NumPy/backpropagation notebook, this time using **PyTorch**. The goal is to
show the same network expressed with a deep-learning framework, where
`nn.Linear` supplies the weights *and* biases and autograd handles the
backward pass that was hand-derived before.

To make the network actually earn its hidden layer, it is trained on the
**two-moons** dataset — two interleaving half-circles that are *not* linearly
separable, so a linear model cannot solve them but a small MLP can.

**Requirements:** `torch`, `scikit-learn`, `numpy`, `matplotlib`, `seaborn`.


# Section 1: Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split


In [ ]:
# Consistent visual style and reproducible seeds
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
torch.manual_seed(0)
np.random.seed(0)


# Section 2: Data Generation & Exploration

The two-moons dataset gives two interleaving classes that no straight line can
separate, so the hidden layer has real work to do. The data is split into a
training set and a held-out test set so we can measure generalisation rather
than just memorisation.


In [ ]:
X, y = make_moons(n_samples=300, noise=0.20, random_state=0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# Convert to tensors. Targets are shaped (N, 1) and float32 to match the
# raw logits produced by the network (required by BCEWithLogitsLoss).
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print(f"Train: {X_train_t.shape[0]} samples   Test: {X_test_t.shape[0]} samples")


In [ ]:
def plot_data(X, y):
    """Scatter the two classes in the 2-D feature space."""
    plt.figure(figsize=(6, 5))
    plt.scatter(X[y == 0, 0], X[y == 0, 1],
                color="steelblue", edgecolor="k", label="Class 0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1],
                color="firebrick", edgecolor="k", label="Class 1")
    plt.title("Two-Moons Dataset", fontweight="bold")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.tight_layout()
    plt.savefig("torch_input_data.png", bbox_inches="tight")
    plt.show()


plot_data(X, y)


## Interpretation:
- The two classes form interleaving crescents that overlap slightly because of
  the added noise.
- No single straight line separates them, so a linear classifier would do
  poorly here — this is exactly the case where a hidden layer matters.


# Section 3: Network Definition

The same architecture as the from-scratch version — input layer, one hidden
layer with a non-linear activation, and a single output node — but expressed
as an `nn.Module`. Two differences from the NumPy implementation:

- `nn.Linear` includes a **bias** term by default (the from-scratch backprop
  network had none).
- The forward pass returns **raw logits**; the sigmoid is folded into
  `BCEWithLogitsLoss` for numerical stability.


In [ ]:
class MLP(nn.Module):
    """One-hidden-layer multi-layer perceptron for binary classification.

    Parameters
    ----------
    input_size : int
        Number of input features.
    hidden_size : int
        Number of hidden nodes.
    output_size : int
        Number of output nodes (1 for binary classification).
    """

    def __init__(self, input_size=2, hidden_size=8, output_size=1):
        super().__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.activation = nn.Tanh()
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.activation(self.hidden(x))
        return self.output(x)  # raw logits; sigmoid applied in the loss


model = MLP(input_size=2, hidden_size=8, output_size=1)
print(model)


# Section 4: Training

Training uses binary cross-entropy on the logits (`BCEWithLogitsLoss`) and the
Adam optimiser. PyTorch's autograd computes every gradient automatically, so
the hand-written backward pass from the NumPy notebook is no longer needed.


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

num_epochs = 500
losses = []

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    logits = model(X_train_t)          # Forward pass
    loss = criterion(logits, y_train_t)
    loss.backward()                    # Autograd backward pass
    optimizer.step()                   # Update weights and biases

    losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:>4}, Loss: {loss.item():.4f}")


# Section 5: Evaluation

In [ ]:
def accuracy(model, X_t, y_t):
    """Fraction of correct predictions at the 0.5 probability threshold."""
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_t))
        preds = (probs >= 0.5).float()
        return (preds == y_t).float().mean().item()


In [ ]:
print("=" * 60)
print("  Training Result")
print("=" * 60)
print(f"  Final loss:      {losses[-1]:.4f}")
print(f"  Train accuracy:  {accuracy(model, X_train_t, y_train_t):.2%}")
print(f"  Test accuracy:   {accuracy(model, X_test_t, y_test_t):.2%}")


In [ ]:
def plot_loss(losses):
    """Plot the binary cross-entropy loss over training."""
    plt.figure(figsize=(6, 4))
    plt.plot(losses, color="steelblue", linewidth=2)
    plt.title("Training Loss", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Binary cross-entropy loss")
    plt.tight_layout()
    plt.savefig("torch_training_loss.png", bbox_inches="tight")
    plt.show()


plot_loss(losses)


## Interpretation:
- The loss falls quickly in the first ~100 epochs, then flattens as the network
  settles on a boundary.
- Train and test accuracy should be close (both high), indicating the network
  generalises to unseen points rather than memorising the training set.


# Section 6: Decision Boundary

Because the data is 2-D, the learned boundary can be drawn directly. The network
is evaluated over a dense grid and the predicted probability is shaded — this is
the non-linear boundary a linear model could never produce.


In [ ]:
def plot_decision_boundary(model, X, y):
    """Shade the predicted probability over the 2-D feature space."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(grid)).reshape(xx.shape).numpy()

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, probs, levels=20, cmap="RdBu", alpha=0.7)
    plt.colorbar(label="P(class 1)")
    plt.scatter(X[y == 0, 0], X[y == 0, 1],
                color="steelblue", edgecolor="k", label="Class 0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1],
                color="firebrick", edgecolor="k", label="Class 1")
    plt.title("Learned Decision Boundary", fontweight="bold")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.tight_layout()
    plt.savefig("torch_decision_boundary.png", bbox_inches="tight")
    plt.show()


plot_decision_boundary(model, X, y)


## Interpretation:
- The shaded regions follow the curved shape of the two moons, showing the
  hidden layer has learned a genuinely non-linear boundary.
- This is the key contrast with a linear classifier and with the 1-D
  from-scratch example: here the MLP's extra capacity is doing real work.

---

### Relationship to the from-scratch version
This network is the framework counterpart to my NumPy/backpropagation
implementation. The NumPy notebook derives and codes the forward and backward
passes by hand; this one delegates the backward pass to autograd and the
weights/biases to `nn.Linear`, while keeping the same overall architecture.
Together they demonstrate both the underlying mechanics and practical PyTorch
usage.
